# Notebook 09 v4 — Post-hoc Re-evaluation on the Previously Inspected 2021–2024 Period

This notebook applies the frozen Stage 07 XGBoost `I_full_40` pipeline and the
frozen Stage 08 daily Top-5% policy to the 2021–2024 evaluation period.

## Scientific status

This period is **not statistically untouched**. Earlier project iterations and
the Breadth design process inspected results from the same 2021–2024 period.
Therefore:

- this run is a post-hoc/exploratory re-evaluation;
- confirmatory or external-validation claims are not allowed;
- the next genuine confirmation must use later data not used in design,
  or prospective paper trading.

No tuning occurs here. Scores and selections are written to an outcome-free
lock before labels and corrected event outcomes are joined.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import joblib
import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("joblib:", joblib.__version__)


## 1. Locate repository and import frozen project modules

In [ ]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.probability_policy import probability_metrics
from src.evaluation.signal_policy import (
    binary_signal_metrics,
    select_daily_top_fraction,
)
from src.evaluation.unseen_test_signal import (
    file_sha256,
    corrected_event_outcome_metrics,
    grouped_corrected_event_outcome_metrics,
    reconstruct_corrected_event_outcomes,
)
from src.features.leakage_checks import (
    ConfirmedZigZagConfig,
    build_candidate_long_mask,
    build_confirmation_gated_zigzag_state,
)
from src.features.preprocessing import (
    CARRIED_STAGE04_NUMERIC_FEATURES,
    EMA_WINDOWS,
    ENGINEERED_MODEL_FEATURES,
    FINAL_CATEGORICAL_FEATURES as BASE_CATEGORICAL_FEATURES,
    FINAL_MODEL_FEATURES as BASE_MODEL_FEATURES,
    FINAL_NUMERIC_FEATURES as BASE_NUMERIC_FEATURES,
    MARKET_INDEX_REQUIRED_COLUMNS,
    RAW_REQUIRED_COLUMNS,
    FeatureEngineeringConfig,
    build_canonical_market_index,
    build_final_feature_frame,
    build_market_regime_feature_frame,
    parse_market_date,
    prepare_market_index_source,
)
from src.features.stage04_breadth_extension import (
    STAGE04_BREADTH_CATEGORICAL_FEATURES,
    STAGE04_BREADTH_FEATURES,
    STAGE04_BREADTH_NUMERIC_FEATURES,
    Stage04BreadthConfig,
    build_daily_market_breadth,
)
from src.features.unseen_breadth import (
    UNSEEN_BREADTH_SCHEMA_VERSION,
    load_started_run_length,
    merge_stage04_breadth_features,
    prepare_symbol_breadth_observations,
)
from src.models.frozen_training import dataframe_fingerprint
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


## 2. Load frozen configuration, model lineage, and policy disclosure


In [ ]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"JSON artifact must be an object: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")
stage09_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "unseen_test_evaluation.yaml"
)

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"
LABELED_TEST_PATH = DATA_PATHS["labeled_data"] / "unseen_test"
RAW_DATA_PATH = DATA_PATHS["raw_data"]

frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
stage04_policy_path = RESULT_PATHS["manifests"] / "04_candidate_filter_policy.json"
stage07_manifest_path = RESULT_PATHS["manifests"] / "07_frozen_model_training_manifest.json"
stage08_policy_path = RESULT_PATHS["manifests"] / "08_probability_signal_policy.json"
stage08_manifest_path = RESULT_PATHS["manifests"] / "08_probability_signal_policy_manifest.json"
model_path = RESULT_PATHS["models"] / "07_frozen_xgboost_pipeline.joblib"

for path in [
    frozen_universe_path,
    stage04_policy_path,
    stage07_manifest_path,
    stage08_policy_path,
    stage08_manifest_path,
    model_path,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required frozen input is missing: {path}")

frozen_universe_df = pd.read_csv(frozen_universe_path, low_memory=False)
stage04_policy = load_json(stage04_policy_path)
stage07_manifest = load_json(stage07_manifest_path)
stage08_policy = load_json(stage08_policy_path)
stage08_manifest = load_json(stage08_manifest_path)

TEST_START = pd.Timestamp(stage09_config["temporal_scope"]["unseen_test_start"])
SIGNAL_GENERATION_END = pd.Timestamp(
    stage09_config["temporal_scope"]["signal_generation_end"]
)
OUTCOME_OBSERVATION_TAIL_END = pd.Timestamp(
    stage09_config["temporal_scope"]["outcome_observation_tail_end"]
)
TEST_END = SIGNAL_GENERATION_END
TRAIN_END = pd.Timestamp(stage09_config["temporal_scope"]["train_end"])

frozen_symbols = sorted(frozen_universe_df["symbol"].astype(str).tolist())
frozen_symbol_set = set(frozen_symbols)

SELECTED_MODEL = str(stage08_policy["score_policy"]["source_model"])
SELECTED_FEATURE_SET = str(
    stage08_policy["score_policy"]["source_feature_set"]
)
SELECTED_TRIAL = int(stage08_policy["score_policy"]["source_trial"])
SELECTED_FEATURES = list(
    stage08_policy["feature_lineage"]["selected_raw_features"]
)
SELECTED_NUMERIC_FEATURES = list(
    stage07_manifest["features"]["numeric_features"]
)
SELECTED_CATEGORICAL_FEATURES = list(
    stage07_manifest["features"]["categorical_features"]
)

assert stage09_config["status"] == (
    "stage_09_configured_v4_breadth_post_hoc_retest"
)
assert stage09_config["scientific_status"][
    "prior_test_period_previously_inspected"
] is True
assert stage09_config["scientific_status"][
    "confirmatory_claim_allowed"
] is False
assert UNSEEN_BREADTH_SCHEMA_VERSION == (
    "stage09_v4_causal_stage04_breadth_reconstruction"
)
assert OUTCOME_OBSERVATION_TAIL_END >= SIGNAL_GENERATION_END
assert len(frozen_symbols) == 499

assert stage04_policy["status"] == "frozen_after_integrity_checks"
assert stage04_policy["primary_side"] == "long"
assert np.isclose(float(stage04_policy["primary_threshold_fraction"]), 0.15)

assert stage07_manifest["status"] == "frozen_after_integrity_checks"
assert stage07_manifest["model"]["selected_model"] == "xgboost"
assert stage07_manifest["model"]["selected_feature_set"] == "I_full_40"
assert stage07_manifest["model"]["selected_trial_number"] == SELECTED_TRIAL
assert stage07_manifest["unseen_test_used"] is False

assert stage08_policy["status"] == "frozen_train_only_policy"
assert stage08_policy["score_policy"]["selected_calibration_method"] == "raw_identity"
assert stage08_policy["score_policy"]["probability_calibrator_fitted"] is False
assert stage08_policy["score_policy"]["literal_probability_interpretation_allowed"] is False
assert np.isclose(float(stage08_policy["signal_policy"]["selected_fraction"]), 0.05)
assert stage08_policy["signal_policy"]["fixed_probability_threshold_selected"] is False
assert stage08_manifest["status"] == "frozen_after_integrity_checks"

assert SELECTED_MODEL == "xgboost"
assert SELECTED_FEATURE_SET == "I_full_40"
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert list(stage07_manifest["features"]["model_features"]) == SELECTED_FEATURES
assert list(stage07_manifest["features"]["numeric_features"]) == SELECTED_NUMERIC_FEATURES
assert list(stage07_manifest["features"]["categorical_features"]) == SELECTED_CATEGORICAL_FEATURES

expected_model_hash = str(stage09_config["frozen_inputs"]["expected_model_sha256"])
actual_model_hash = file_sha256(model_path)
assert actual_model_hash == expected_model_hash
assert actual_model_hash == stage07_manifest["model"]["model_sha256"]

lineage_audit_df = pd.DataFrame([{
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_claim_allowed": False,
    "stage04_candidate_threshold_fraction": float(stage04_policy["primary_threshold_fraction"]),
    "stage07_code_sha": stage07_manifest["git_commit_sha"],
    "stage07_model_sha256": actual_model_hash,
    "stage07_selected_model": SELECTED_MODEL,
    "stage07_selected_feature_set": SELECTED_FEATURE_SET,
    "stage07_selected_trial": SELECTED_TRIAL,
    "stage07_raw_features": len(SELECTED_FEATURES),
    "stage08_code_sha": stage08_manifest["git_commit_sha"],
    "stage08_calibration_method": stage08_policy["score_policy"]["selected_calibration_method"],
    "stage08_signal_fraction": float(stage08_policy["signal_policy"]["selected_fraction"]),
    "fixed_probability_threshold_selected": False,
    "model_retrained": False,
    "test_outcomes_used_before_score_lock": False,
}])
lineage_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_frozen_input_lineage_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(lineage_audit_df)


## 3. Validate train/test/raw file inventories

In [ ]:
def symbol_from_path(path: Path, suffix: str) -> str:
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected file name: {path.name}")
    return path.name[:-len(suffix)]


train_files = sorted(LABELED_TRAIN_PATH.glob("*_train_labeled.csv"))
test_files = sorted(LABELED_TEST_PATH.glob("*_unseen_test_labeled.csv"))
raw_files = sorted(RAW_DATA_PATH.glob("*.csv"))

train_file_map = {
    symbol_from_path(path, "_train_labeled.csv"): path
    for path in train_files
}
test_file_map = {
    symbol_from_path(path, "_unseen_test_labeled.csv"): path
    for path in test_files
}
raw_file_map = {path.stem: path for path in raw_files}

assert set(train_file_map) == frozen_symbol_set
assert set(test_file_map) == frozen_symbol_set
assert set(raw_file_map).issuperset(frozen_symbol_set)

inventory_audit_df = pd.DataFrame([{
    "frozen_symbols": len(frozen_symbols),
    "labeled_train_files": len(train_file_map),
    "labeled_unseen_test_files": len(test_file_map),
    "raw_files_covering_frozen_universe": int(len(frozen_symbol_set.intersection(raw_file_map))),
    "test_start": TEST_START,
    "signal_generation_end": SIGNAL_GENERATION_END,
    "outcome_observation_tail_end": OUTCOME_OBSERVATION_TAIL_END,
}])
inventory_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_inventory_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(inventory_audit_df)


## 4. Build causal market-index and Stage 04 Breadth history through the signal end

Both market feature families use raw observations no later than
22 September 2024. Breadth is reconstructed from previous-valid adjusted-close
returns over the frozen 499-symbol universe using the exact Stage 04 formula.


In [ ]:
feature_config = FeatureEngineeringConfig(
    relative_strength_window=12,
    return_zscore_window=12,
    volume_window=30,
    market_volatility_window=20,
    market_ema_fast_window=20,
    market_ema_slow_window=60,
    market_drawdown_window=60,
    market_index_consistency_relative_tolerance=1.0e-10,
)
breadth_config = Stage04BreadthConfig()

market_index_parts = []
breadth_observation_parts = []
market_index_error_rows = []
market_rows_after_test_end = 0
started = time.perf_counter()

for file_number, symbol in enumerate(frozen_symbols, start=1):
    raw_path = raw_file_map[symbol]
    try:
        raw_market = pd.read_csv(
            raw_path,
            usecols=list(MARKET_INDEX_REQUIRED_COLUMNS),
            low_memory=False,
        )
        raw_dates = parse_market_date(raw_market["dEven"])
        market_rows_after_test_end += int(raw_dates.gt(TEST_END).sum())
        raw_market = raw_market.loc[
            raw_dates.notna() & raw_dates.le(TEST_END)
        ].copy()
        market_index_parts.append(
            prepare_market_index_source(raw_market, source_symbol=symbol)
        )
        breadth_observation_parts.append(
            prepare_symbol_breadth_observations(
                raw_path,
                symbol=symbol,
                horizon_end=TEST_END,
                config=breadth_config,
            )
        )
    except Exception as exc:
        market_index_error_rows.append({
            "symbol": symbol,
            "file_name": raw_path.name,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

    if file_number == 1 or file_number % 50 == 0 or file_number == len(frozen_symbols):
        print(
            f"[market history] [{file_number:>4}/{len(frozen_symbols)}] "
            f"elapsed={time.perf_counter() - started:,.1f}s "
            f"errors={len(market_index_error_rows)}"
        )

market_index_errors_df = pd.DataFrame(
    market_index_error_rows,
    columns=["symbol", "file_name", "error_type", "error_message"],
)
market_index_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_market_index_source_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not market_index_errors_df.empty:
    raise RuntimeError("Market-index or Breadth source errors exist.")

market_observation_panel = pd.concat(market_index_parts, ignore_index=True)
canonical_market_index_df, market_consistency_audit_df = build_canonical_market_index(
    market_observation_panel,
    relative_tolerance=feature_config.market_index_consistency_relative_tolerance,
)
market_regime_feature_frame, market_regime_audit = build_market_regime_feature_frame(
    canonical_market_index_df,
    config=feature_config,
)

breadth_observation_panel = pd.concat(
    breadth_observation_parts,
    ignore_index=True,
)
market_breadth_feature_frame, breadth_audit = build_daily_market_breadth(
    breadth_observation_panel,
    config=breadth_config,
)

market_consistency_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_market_index_consistency_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
market_regime_audit_df = pd.DataFrame([{
    **market_regime_audit,
    "test_feature_horizon_end": TEST_END,
    "raw_rows_excluded_after_test_end": market_rows_after_test_end,
    "inconsistent_cross_file_rows": int(
        market_consistency_audit_df["inconsistent_across_raw_files"].sum()
    ),
}])
market_regime_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_market_regime_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

breadth_audit_df = pd.DataFrame([{
    **breadth_audit,
    "schema_version": UNSEEN_BREADTH_SCHEMA_VERSION,
    "frozen_symbols": len(frozen_symbols),
    "feature_horizon_end": TEST_END,
    "same_day_or_past_only": True,
    "future_rows_used": False,
}])
breadth_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_market_breadth_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert canonical_market_index_df["dEven"].max() <= TEST_END
assert market_regime_feature_frame["dEven"].max() <= TEST_END
assert market_regime_feature_frame["dEven"].is_unique
assert market_regime_feature_frame["dEven"].is_monotonic_increasing

assert market_breadth_feature_frame["dEven"].max() <= TEST_END
assert market_breadth_feature_frame["dEven"].is_unique
assert market_breadth_feature_frame["dEven"].is_monotonic_increasing
assert market_breadth_feature_frame["market_breadth_raw"].between(
    -1.0, 1.0, inclusive="both"
).all()
assert market_breadth_feature_frame[
    [
        "market_breadth_raw",
        "market_breadth_ema30",
        "market_breadth_slope5",
        "market_breadth_regime",
    ]
].notna().all().all()

display(market_regime_audit_df)
display(breadth_audit_df)


## 5. Build causal 40-feature candidate rows without loading outcomes

The train history is prepended as rolling warm-up. The original `started`
run length is loaded without recomputation. The five Stage 04 additions are
merged after the unchanged +15% ZigZag candidate rule is applied.


In [ ]:
zigzag_policy = columns_config["zigzag_audit"]
zigzag_config = ConfirmedZigZagConfig(
    depth=int(zigzag_policy["source_depth"]),
    deviation_percent=float(zigzag_policy["source_deviation_percent"]),
)
PRIMARY_THRESHOLD = float(stage04_policy["primary_threshold_fraction"])

history_feature_columns = [
    "dEven", "adj_high", "adj_low", "adj_last_price", "RSI_14", "macd",
    *[f"EMA_{window}" for window in EMA_WINDOWS],
]
test_inference_columns = [*history_feature_columns, "eligible_for_modeling"]

candidate_parts = []
feature_audit_rows = []
candidate_audit_rows = []
candidate_error_rows = []
started_source_counts = {}
total_test_rows = 0
total_test_eligible = 0
started = time.perf_counter()

for file_number, symbol in enumerate(frozen_symbols, start=1):
    train_path = train_file_map[symbol]
    test_path = test_file_map[symbol]
    raw_path = raw_file_map[symbol]
    try:
        train_history = pd.read_csv(
            train_path,
            usecols=history_feature_columns,
            low_memory=False,
        )
        test_history = pd.read_csv(
            test_path,
            usecols=test_inference_columns,
            low_memory=False,
        )
        train_history["partition"] = "train"
        train_history["eligible_for_modeling"] = False
        test_history["partition"] = "unseen_test"

        full_history = pd.concat([train_history, test_history], ignore_index=True)
        full_history["dEven"] = pd.to_datetime(
            full_history["dEven"], errors="raise"
        ).dt.normalize()
        full_history = full_history.sort_values(
            ["dEven", "partition"], kind="stable"
        ).drop_duplicates(
            subset=["dEven"], keep="last"
        ).reset_index(drop=True)
        full_history = full_history.loc[
            full_history["dEven"].le(TEST_END)
        ].reset_index(drop=True)

        test_mask = full_history["partition"].eq("unseen_test")
        symbol_test_rows = int(test_mask.sum())
        symbol_test_eligible = int(
            full_history.loc[test_mask, "eligible_for_modeling"]
            .fillna(False).astype(bool).sum()
        )
        total_test_rows += symbol_test_rows
        total_test_eligible += symbol_test_eligible

        raw_frame = pd.read_csv(
            raw_path,
            usecols=list(RAW_REQUIRED_COLUMNS),
            low_memory=False,
        )
        raw_dates = parse_market_date(raw_frame["dEven"])
        raw_frame = raw_frame.loc[
            raw_dates.notna() & raw_dates.le(TEST_END)
        ].copy()

        final_feature_frame, feature_audit = build_final_feature_frame(
            full_history[history_feature_columns],
            raw_frame,
            market_regime_feature_frame,
            config=feature_config,
        )
        zigzag_state = build_confirmation_gated_zigzag_state(
            full_history[["dEven", "adj_high", "adj_low", "adj_last_price"]],
            config=zigzag_config,
            date_column="dEven",
        )

        eligible_series = full_history["eligible_for_modeling"].fillna(False).astype(bool)
        candidate_mask = build_candidate_long_mask(
            zigzag_state,
            eligible_for_modeling=eligible_series,
            threshold_fraction=PRIMARY_THRESHOLD,
        ) & test_mask

        candidate_dates = full_history.loc[candidate_mask, "dEven"].reset_index(drop=True)
        state_view = zigzag_state.loc[candidate_mask].reset_index(drop=True)
        feature_view = final_feature_frame.loc[
            final_feature_frame["dEven"].isin(candidate_dates)
        ].sort_values("dEven", kind="stable").reset_index(drop=True)

        candidate_frame = pd.DataFrame({
            "event_id": symbol + "::" + candidate_dates.dt.strftime("%Y-%m-%d"),
            "symbol": symbol,
            "dEven": candidate_dates,
        })
        candidate_frame[list(CARRIED_STAGE04_NUMERIC_FEATURES)] = (
            state_view[list(CARRIED_STAGE04_NUMERIC_FEATURES)]
        )
        candidate_frame = candidate_frame.merge(
            feature_view[["dEven", *ENGINEERED_MODEL_FEATURES]],
            on="dEven",
            how="left",
            validate="one_to_one",
        )

        started_frame, started_source_name = load_started_run_length(
            symbol=symbol,
            raw_path=raw_path,
            fallback_path=test_path,
            horizon_end=TEST_END,
            config=breadth_config,
        )
        started_source_counts[started_source_name] = (
            started_source_counts.get(started_source_name, 0) + 1
        )
        candidate_frame, breadth_merge_audit = merge_stage04_breadth_features(
            candidate_frame,
            started_frame=started_frame,
            breadth_frame=market_breadth_feature_frame,
            config=breadth_config,
        )

        missing_features = sorted(
            set(SELECTED_FEATURES) - set(candidate_frame.columns)
        )
        if missing_features:
            raise KeyError(
                f"Selected candidate features are missing: {missing_features}"
            )

        candidate_frame = candidate_frame[
            ["event_id", "symbol", "dEven", *SELECTED_FEATURES]
        ].copy()
        candidate_parts.append(candidate_frame)

        numeric_array = candidate_frame[
            SELECTED_NUMERIC_FEATURES
        ].to_numpy(dtype=float)

        feature_audit_rows.append({
            "symbol": symbol,
            "full_history_rows": int(len(full_history)),
            "test_rows": symbol_test_rows,
            "test_eligible_events": symbol_test_eligible,
            "candidate_events": int(len(candidate_frame)),
            "candidate_rows_with_any_feature_missing": int(
                candidate_frame[SELECTED_FEATURES].isna().any(axis=1).sum()
            ),
            "candidate_nonfinite_numeric_values": int(
                np.isinf(numeric_array).sum()
            ),
            "started_source": started_source_name,
            "breadth_identity_preserved": bool(
                breadth_merge_audit["candidate_identity_preserved"]
            ),
            **feature_audit,
        })
        candidate_audit_rows.append({
            "symbol": symbol,
            "test_rows": symbol_test_rows,
            "test_eligible_events": symbol_test_eligible,
            "candidate_events": int(len(candidate_frame)),
            "first_candidate_date": candidate_frame["dEven"].min() if len(candidate_frame) else pd.NaT,
            "last_candidate_date": candidate_frame["dEven"].max() if len(candidate_frame) else pd.NaT,
        })

    except Exception as exc:
        candidate_error_rows.append({
            "symbol": symbol,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

    if file_number == 1 or file_number % 25 == 0 or file_number == len(frozen_symbols):
        print(
            f"[test candidates] [{file_number:>4}/{len(frozen_symbols)}] "
            f"elapsed={time.perf_counter() - started:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=["symbol", "error_type", "error_message"],
)
candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_candidate_generation_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not candidate_errors_df.empty:
    display(candidate_errors_df.head(20))
    raise RuntimeError("Unseen-period candidate generation errors exist.")

candidate_panel = pd.concat(candidate_parts, ignore_index=True).sort_values(
    ["dEven", "symbol", "event_id"], kind="stable"
).reset_index(drop=True)
feature_engineering_audit_df = pd.DataFrame(feature_audit_rows)
candidate_by_symbol_audit_df = pd.DataFrame(candidate_audit_rows)
feature_engineering_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_feature_engineering_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
candidate_by_symbol_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_candidate_by_symbol_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert total_test_rows == int(stage09_config["frozen_inputs"]["expected_unseen_test_rows"])
assert total_test_eligible == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_eligible_events"]
)
assert len(candidate_panel) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert candidate_panel["event_id"].is_unique
assert candidate_panel["dEven"].between(TEST_START, TEST_END).all()
assert len(BASE_MODEL_FEATURES) == 35
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert set(STAGE04_BREADTH_FEATURES).issubset(SELECTED_FEATURES)
assert candidate_panel[SELECTED_FEATURES].notna().all().all()
assert not np.isinf(
    candidate_panel[SELECTED_NUMERIC_FEATURES].to_numpy(dtype=float)
).any()

candidate_fingerprint = dataframe_fingerprint(
    candidate_panel,
    ["event_id", "symbol", "dEven", *SELECTED_FEATURES],
)
candidate_panel_audit_df = pd.DataFrame([{
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "unseen_test_rows": total_test_rows,
    "unseen_test_eligible_events": total_test_eligible,
    "candidate_events": int(len(candidate_panel)),
    "symbols_with_candidates": int(candidate_panel["symbol"].nunique()),
    "signal_dates": int(candidate_panel["dEven"].nunique()),
    "first_candidate_date": candidate_panel["dEven"].min(),
    "last_candidate_date": candidate_panel["dEven"].max(),
    "duplicate_event_ids": int(candidate_panel["event_id"].duplicated().sum()),
    "selected_feature_set": SELECTED_FEATURE_SET,
    "raw_model_features": len(SELECTED_FEATURES),
    "numeric_model_features": len(SELECTED_NUMERIC_FEATURES),
    "categorical_model_features": len(SELECTED_CATEGORICAL_FEATURES),
    "started_source_counts": json.dumps(started_source_counts, sort_keys=True),
    "candidate_population_sha256": candidate_fingerprint,
    "outcome_columns_loaded": False,
}])
candidate_panel_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(candidate_panel_audit_df)


## 6. Frozen 40-feature inference and outcome-free policy lock

Scores and selections are written before any test label or event-outcome column
is read. No expected lock hash or expected performance metric is supplied.


In [ ]:
frozen_pipeline = joblib.load(model_path)
transformed_feature_names = [
    str(value)
    for value in frozen_pipeline.named_steps["preprocess"].get_feature_names_out()
]

assert len(transformed_feature_names) == int(
    stage09_config["frozen_inputs"]["expected_transformed_features"]
)
assert len(transformed_feature_names) == 47

inference_started = time.perf_counter()
raw_scores = frozen_pipeline.predict_proba(
    candidate_panel[SELECTED_FEATURES]
)[:, 1]
inference_seconds = time.perf_counter() - inference_started

assert np.isfinite(raw_scores).all()
assert ((raw_scores >= 0.0) & (raw_scores <= 1.0)).all()

inference_frame = candidate_panel[
    ["event_id", "symbol", "dEven"]
].copy()
inference_frame["xgboost_ranking_score"] = raw_scores

selected_fraction = float(
    stage08_policy["signal_policy"]["selected_fraction"]
)
minimum_per_date = int(
    stage08_policy["signal_policy"]["minimum_signals_per_date"]
)

inference_lock_df = select_daily_top_fraction(
    inference_frame,
    score_column="xgboost_ranking_score",
    date_column="dEven",
    fraction=selected_fraction,
    minimum_per_date=minimum_per_date,
    symbol_column="symbol",
    event_id_column="event_id",
)

lock_columns = [
    "event_id",
    "symbol",
    "dEven",
    "xgboost_ranking_score",
    "daily_candidate_count",
    "daily_rank",
    "daily_signal_quota",
    "daily_score_cutoff",
    "selected_signal",
]
lock_path = (
    RESULT_PATHS["predictions"]
    / "09_unseen_test_inference_lock.csv"
)
inference_lock_df[lock_columns].to_csv(
    lock_path,
    index=False,
    encoding="utf-8-sig",
)

lock_file_hash = file_sha256(lock_path)

assert len(inference_lock_df) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert inference_lock_df["dEven"].nunique() == int(
    stage09_config["frozen_inputs"]["expected_signal_dates"]
)
assert int(inference_lock_df["selected_signal"].sum()) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)
assert inference_lock_df["selected_signal"].sum() == (
    inference_lock_df.groupby("dEven")[
        "daily_signal_quota"
    ].first().sum()
)
assert inference_lock_df.groupby("dEven")[
    "selected_signal"
].sum().ge(1).all()

inference_lock_manifest = {
    "stage": 9,
    "status": "outcome_free_inference_locked",
    "scientific_status": (
        "post_hoc_retest_previously_inspected_period"
    ),
    "confirmatory_claim_allowed": False,
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "model_sha256": actual_model_hash,
    "selected_model": SELECTED_MODEL,
    "selected_feature_set": SELECTED_FEATURE_SET,
    "selected_trial": SELECTED_TRIAL,
    "raw_feature_count": len(SELECTED_FEATURES),
    "transformed_feature_count": len(transformed_feature_names),
    "candidate_population_sha256": candidate_fingerprint,
    "inference_lock_file": str(lock_path),
    "inference_lock_file_sha256": lock_file_hash,
    "candidate_events": int(len(inference_lock_df)),
    "signal_dates": int(inference_lock_df["dEven"].nunique()),
    "selected_signals": int(inference_lock_df["selected_signal"].sum()),
    "selected_fraction_rule": selected_fraction,
    "minimum_signals_per_date": minimum_per_date,
    "ranking_order": stage08_policy["signal_policy"]["ranking_order"],
    "calibration_method": "raw_identity",
    "fixed_probability_threshold_selected": False,
    "outcome_columns_in_lock_file": False,
    "test_labels_loaded_before_lock": False,
    "test_outcomes_loaded_before_lock": False,
    "model_retrained": False,
    "inference_seconds": float(inference_seconds),
}

lock_manifest_path = (
    RESULT_PATHS["manifests"]
    / "09_unseen_test_inference_lock.json"
)
lock_manifest_path.write_text(
    json.dumps(
        inference_lock_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Outcome-free inference lock:", lock_path)
print("Lock SHA256:", lock_file_hash)
print("Candidate events:", len(inference_lock_df))
print(
    "Selected signals:",
    int(inference_lock_df["selected_signal"].sum()),
)
print("Test labels loaded before lock: False")
print("Expected performance metrics supplied before lock: False")


## 7. Open outcomes and reconstruct corrected event returns after the lock

Signal dates remain capped at 22 September 2024. Raw observations through
26 October 2024 may only complete the already-defined 30-observation outcome
window. No post-cutoff row is used for features, scores, or signal selection.

In [ ]:
outcome_columns = [
    "dEven", "eligible_for_modeling", "label", "event_end_date",
    "event_return", "holding_period_observations", "barrier_touched",
    "label_status", "same_bar_double_touch",
]
outcome_parts = []
outcome_error_rows = []
candidate_event_ids = set(inference_lock_df["event_id"].astype(str))
selected_event_ids = set(
    inference_lock_df.loc[
        inference_lock_df["selected_signal"].astype(bool), "event_id"
    ].astype(str)
)

for symbol in frozen_symbols:
    path = test_file_map[symbol]
    try:
        frame = pd.read_csv(path, usecols=outcome_columns, low_memory=False)
        frame["dEven"] = pd.to_datetime(
            frame["dEven"], errors="raise"
        ).dt.normalize()
        frame = frame.loc[
            frame["eligible_for_modeling"].fillna(False).astype(bool)
        ].copy()
        frame["event_id"] = symbol + "::" + frame["dEven"].dt.strftime("%Y-%m-%d")
        frame["symbol"] = symbol
        frame = frame.loc[frame["event_id"].isin(candidate_event_ids)].copy()
        outcome_parts.append(frame)
    except Exception as exc:
        outcome_error_rows.append({
            "symbol": symbol,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

outcome_errors_df = pd.DataFrame(
    outcome_error_rows,
    columns=["symbol", "error_type", "error_message"],
)
outcome_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_outcome_join_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not outcome_errors_df.empty:
    raise RuntimeError("Unseen-test outcome-loading errors exist.")

outcome_panel = pd.concat(outcome_parts, ignore_index=True)
outcome_panel["meta_label"] = pd.to_numeric(
    outcome_panel["label"], errors="raise"
).astype(int)
outcome_panel["event_end_date"] = pd.to_datetime(
    outcome_panel["event_end_date"], errors="raise"
).dt.normalize()
assert outcome_panel["event_id"].is_unique
assert set(outcome_panel["event_id"].astype(str)) == candidate_event_ids
assert outcome_panel["dEven"].le(SIGNAL_GENERATION_END).all()
assert outcome_panel["event_end_date"].le(SIGNAL_GENERATION_END).all()

# Corrected economic outcomes are reconstructed only after the frozen
# outcome-free score and signal lock exists. Predictive and signal-
# classification metrics use the complete unchanged candidate population.
selected_outcome_input = outcome_panel.loc[
    outcome_panel["event_id"].isin(selected_event_ids),
    [
        "event_id", "symbol", "dEven", "meta_label",
        "barrier_touched", "event_return", "event_end_date",
    ],
].copy()
assert len(selected_outcome_input) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)
assert selected_outcome_input["event_id"].is_unique
assert set(selected_outcome_input["event_id"].astype(str)) == selected_event_ids

corrected_outcome_panel, corrected_outcome_errors_df = (
    reconstruct_corrected_event_outcomes(
        selected_outcome_input,
        raw_file_map=raw_file_map,
        signal_generation_end=SIGNAL_GENERATION_END,
        outcome_observation_tail_end=OUTCOME_OBSERVATION_TAIL_END,
        horizon_observations=int(
            stage09_config["corrected_event_outcome_policy"][
                "horizon_observations"
            ]
        ),
        upper_barrier_return=float(
            stage09_config["corrected_event_outcome_policy"][
                "upper_barrier_return"
            ]
        ),
        lower_barrier_return=float(
            stage09_config["corrected_event_outcome_policy"][
                "lower_barrier_return"
            ]
        ),
    )
)
corrected_outcome_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_corrected_event_return_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not corrected_outcome_errors_df.empty:
    display(
        corrected_outcome_errors_df.groupby(
            ["error_type", "error_message"], dropna=False
        ).size().reset_index(name="events").head(20)
    )
    raise RuntimeError("Corrected selected-signal return reconstruction errors exist.")

assert corrected_outcome_panel["event_id"].is_unique
assert set(corrected_outcome_panel["event_id"].astype(str)) == selected_event_ids

evaluation_df = inference_lock_df[lock_columns].merge(
    outcome_panel[[
        "event_id", "meta_label", "event_end_date", "event_return",
        "holding_period_observations", "barrier_touched", "label_status",
        "same_bar_double_touch",
    ]],
    on="event_id",
    how="left",
    validate="one_to_one",
).merge(
    corrected_outcome_panel,
    on="event_id",
    how="left",
    validate="one_to_one",
)
evaluation_df["calendar_year"] = evaluation_df["dEven"].dt.year.astype(int)
evaluation_df = evaluation_df.rename(
    columns={"event_return": "original_event_return"}
)

assert evaluation_df["meta_label"].isin([0, 1]).all()
assert evaluation_df["label_status"].eq("labeled").all()
assert evaluation_df["original_event_return"].notna().all()
assert evaluation_df["event_end_date"].ge(evaluation_df["dEven"]).all()
assert evaluation_df["event_end_date"].le(SIGNAL_GENERATION_END).all()

selected_evaluation_df = evaluation_df.loc[
    evaluation_df["selected_signal"].astype(bool)
].copy()
assert len(selected_evaluation_df) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)
assert selected_evaluation_df["corrected_event_return"].notna().all()
assert selected_evaluation_df["outcome_window_end_date"].le(
    OUTCOME_OBSERVATION_TAIL_END
).all()
assert (
    selected_evaluation_df["corrected_event_return"].gt(0.0).astype(int)
    == selected_evaluation_df["meta_label"].astype(int)
).all()

full_output_path = (
    RESULT_PATHS["predictions"] / "09_unseen_test_signal_evaluation.csv"
)
selected_output_path = (
    RESULT_PATHS["predictions"] / "09_selected_unseen_test_signals.csv"
)
evaluation_df.to_csv(full_output_path, index=False, encoding="utf-8-sig")
selected_evaluation_df.to_csv(
    selected_output_path,
    index=False,
    encoding="utf-8-sig",
)

selected_tail_mask = selected_evaluation_df[
    "outcome_window_uses_tail"
].astype(bool)
outcome_tail_audit_df = pd.DataFrame([{
    "signal_generation_start": TEST_START,
    "signal_generation_end": SIGNAL_GENERATION_END,
    "outcome_observation_tail_end": OUTCOME_OBSERVATION_TAIL_END,
    "candidate_events_for_predictive_evaluation": int(len(evaluation_df)),
    "corrected_outcome_reconstruction_population": (
        "frozen_policy_selected_signals_only"
    ),
    "corrected_outcome_events": int(len(selected_evaluation_df)),
    "selected_signals": int(evaluation_df["selected_signal"].sum()),
    "selected_signals_using_outcome_tail": int(selected_tail_mask.sum()),
    "latest_signal_date": evaluation_df["dEven"].max(),
    "latest_required_selected_outcome_window_end": (
        selected_evaluation_df["outcome_window_end_date"].max()
    ),
    "signals_generated_after_frozen_end": int(
        evaluation_df["dEven"].gt(SIGNAL_GENERATION_END).sum()
    ),
    "selected_outcome_windows_after_frozen_tail": int(
        selected_evaluation_df["outcome_window_end_date"]
        .gt(OUTCOME_OBSERVATION_TAIL_END).sum()
    ),
    "post_signal_end_rows_used_for_features_or_scoring": False,
}])
outcome_tail_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_outcome_observation_tail_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Candidate outcome labels joined:", len(evaluation_df))
print("Corrected selected-signal outcomes reconstructed:", len(selected_evaluation_df))
print("Selected signals using the outcome tail:", int(selected_tail_mask.sum()))
print(
    "Latest required selected outcome-window date:",
    selected_evaluation_df["outcome_window_end_date"].max().date(),
)


## 8. Post-hoc predictive, signal-classification, and corrected outcome metrics

These metrics describe the frozen retest. They are not confirmatory evidence
because the same calendar period informed earlier project decisions.


In [ ]:
predictive_metric_values = probability_metrics(
    evaluation_df["meta_label"].to_numpy(dtype=int),
    evaluation_df["xgboost_ranking_score"].to_numpy(dtype=float),
    ece_bins=10,
)
predictive_metrics_df = pd.DataFrame([{
    **predictive_metric_values,
    "events": int(len(evaluation_df)),
    "score_interpretation": "uncalibrated_ranking_score",
    "literal_probability_interpretation_allowed": False,
    "calibration_fitted": False,
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_evaluation": False,
}])
predictive_metrics_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_predictive_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_classification = binary_signal_metrics(
    evaluation_df["meta_label"].to_numpy(dtype=int),
    evaluation_df["selected_signal"].to_numpy(dtype=bool),
)
signal_classification_metrics_df = pd.DataFrame([{
    **signal_classification,
    "policy_type": "daily_cross_sectional_top_fraction",
    "selected_fraction_rule": selected_fraction,
    "minimum_signals_per_date": minimum_per_date,
    "fixed_probability_threshold_selected": False,
    "policy_changed_after_test_outcomes": False,
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_claim_allowed": False,
}])
signal_classification_metrics_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_signal_classification_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_outcome_metrics = corrected_event_outcome_metrics(
    selected_evaluation_df
)
signal_outcome_summary_df = pd.DataFrame([{
    "population": "frozen_policy_selected_signals",
    **selected_outcome_metrics,
}])
signal_outcome_summary_df["return_interpretation"] = (
    "corrected_gross_event_level_outcome_not_portfolio_return"
)
signal_outcome_summary_df["upper_return_is_ex_post_maximum"] = True
signal_outcome_summary_df["transaction_costs_applied"] = False
signal_outcome_summary_df["portfolio_backtest_performed"] = False
signal_outcome_summary_df["scientific_status"] = (
    "post_hoc_retest_previously_inspected_period"
)
signal_outcome_summary_df["confirmatory_claim_allowed"] = False
signal_outcome_summary_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_signal_outcome_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_outcomes_by_year_df = grouped_corrected_event_outcome_metrics(
    selected_evaluation_df,
    group_column="calendar_year",
)
signal_outcomes_by_year_df["selected_fraction_rule"] = selected_fraction
signal_outcomes_by_year_df["portfolio_backtest_performed"] = False
signal_outcomes_by_year_df["confirmatory_claim_allowed"] = False
signal_outcomes_by_year_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_signal_outcomes_by_year.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_outcomes_by_barrier_df = grouped_corrected_event_outcome_metrics(
    selected_evaluation_df,
    group_column="barrier_touched",
)
signal_outcomes_by_barrier_df["portfolio_backtest_performed"] = False
signal_outcomes_by_barrier_df["confirmatory_claim_allowed"] = False
signal_outcomes_by_barrier_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_signal_outcomes_by_barrier.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_outcome_row = signal_outcome_summary_df.iloc[0]

# No expected performance value is asserted. Only finite, internally
# consistent outputs are required after the outcome-free lock.
for metric_name in [
    "win_rate",
    "average_winning_return",
    "average_losing_return",
    "payoff_ratio",
    "profit_factor",
    "mean_corrected_event_return",
    "median_corrected_event_return",
]:
    assert np.isfinite(float(selected_outcome_row[metric_name])), metric_name

assert int(selected_outcome_row["events"]) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)
assert int(
    selected_outcome_row["winning_events"]
    + selected_outcome_row["losing_events_negative_return"]
    + selected_outcome_row["breakeven_events"]
) == int(selected_outcome_row["events"])

display(predictive_metrics_df)
display(signal_classification_metrics_df)
display(signal_outcome_summary_df)
display(signal_outcomes_by_year_df)
display(signal_outcomes_by_barrier_df)


## 9. Daily implementation audit

In [ ]:
signal_date_audit_df = evaluation_df.groupby("dEven", as_index=False).agg(
    candidate_events=("event_id", "size"),
    required_signal_quota=("daily_signal_quota", "first"),
    selected_signals=("selected_signal", "sum"),
    score_cutoff=("daily_score_cutoff", "first"),
    minimum_score=("xgboost_ranking_score", "min"),
    maximum_score=("xgboost_ranking_score", "max"),
    selected_positive_labels=(
        "meta_label",
        lambda series: int(series[
            evaluation_df.loc[series.index, "selected_signal"].to_numpy(dtype=bool)
        ].sum()),
    ),
)
daily_selected_return_df = (
    selected_evaluation_df.groupby("dEven", as_index=False)
    .agg(
        selected_mean_corrected_event_return=(
            "corrected_event_return", "mean"
        )
    )
)
signal_date_audit_df = signal_date_audit_df.merge(
    daily_selected_return_df,
    on="dEven",
    how="left",
    validate="one_to_one",
)
signal_date_audit_df["selected_precision"] = (
    signal_date_audit_df["selected_positive_labels"]
    / signal_date_audit_df["selected_signals"]
)
assert signal_date_audit_df["selected_signals"].eq(
    signal_date_audit_df["required_signal_quota"]
).all()
assert signal_date_audit_df["selected_signals"].ge(1).all()
assert signal_date_audit_df[
    "selected_mean_corrected_event_return"
].notna().all()
signal_date_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_unseen_test_signal_date_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(signal_date_audit_df.head())


## 10. Write the Stage 09 v4 post-hoc retest manifest


In [ ]:
selected_outcome_row = signal_outcome_summary_df.iloc[0]

manifest = {
    "stage": 9,
    "status": "completed_internal_integrity_checks",
    "stage_version": "v4_breadth_post_hoc_retest_selected_signal_corrected_outcomes",
    "notebook": "09_signal_level_evaluation.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "scientific_status": {
        "classification": "post_hoc_retest_previously_inspected_period",
        "prior_test_period_previously_inspected": True,
        "breadth_design_informed_by_prior_test_diagnostics": True,
        "confirmatory_claim_allowed": False,
        "external_validation_claim_allowed": False,
        "future_confirmation_required": True,
        "recommended_confirmation": (
            "later data not used in design or prospective paper trading"
        ),
    },
    "lineage": {
        "stage04_candidate_threshold_fraction": PRIMARY_THRESHOLD,
        "stage07_code_commit_sha": stage07_manifest["git_commit_sha"],
        "stage07_model_sha256": actual_model_hash,
        "stage07_selected_model": SELECTED_MODEL,
        "stage07_selected_feature_set": SELECTED_FEATURE_SET,
        "stage07_selected_trial": SELECTED_TRIAL,
        "stage07_selected_raw_features": SELECTED_FEATURES,
        "stage08_code_commit_sha": stage08_manifest["git_commit_sha"],
        "stage08_policy_configuration_hash": stage08_manifest[
            "configuration_hash"
        ],
    },
    "temporal_scope": {
        "evaluation_period_start": str(TEST_START.date()),
        "signal_generation_end": str(SIGNAL_GENERATION_END.date()),
        "outcome_observation_tail_end": str(
            OUTCOME_OBSERVATION_TAIL_END.date()
        ),
        "signals_generated_in_outcome_tail": 0,
    },
    "candidate_population": {
        "events": int(len(evaluation_df)),
        "symbols": int(evaluation_df["symbol"].nunique()),
        "signal_dates": int(evaluation_df["dEven"].nunique()),
        "candidate_population_sha256": candidate_fingerprint,
        "selected_feature_set": SELECTED_FEATURE_SET,
        "raw_feature_count": len(SELECTED_FEATURES),
        "numeric_feature_count": len(SELECTED_NUMERIC_FEATURES),
        "categorical_feature_count": len(SELECTED_CATEGORICAL_FEATURES),
    },
    "inference_lock": inference_lock_manifest,
    "corrected_event_outcome_policy": (
        stage09_config["corrected_event_outcome_policy"]
    ),
    "corrected_outcome_reconstruction_population": (
        "frozen_policy_selected_signals_only"
    ),
    "outcome_tail_audit": outcome_tail_audit_df.iloc[0].to_dict(),
    "predictive_metrics": predictive_metrics_df.iloc[0].to_dict(),
    "signal_classification_metrics": (
        signal_classification_metrics_df.iloc[0].to_dict()
    ),
    "selected_signal_corrected_outcomes": selected_outcome_row.to_dict(),
    "policy": {
        "score_interpretation": "uncalibrated_ranking_score",
        "calibration_method": "raw_identity",
        "calibrator_fitted": False,
        "policy_type": "daily_cross_sectional_top_fraction",
        "selected_fraction": selected_fraction,
        "minimum_signals_per_date": minimum_per_date,
        "fixed_probability_threshold_selected": False,
        "ranking_order": stage08_policy["signal_policy"]["ranking_order"],
    },
    "safeguards": {
        "scores_and_signals_locked_before_outcome_join": True,
        "expected_performance_metrics_supplied_before_lock": False,
        "expected_inference_lock_hash_supplied_before_run": False,
        "test_labels_used_before_inference_lock": False,
        "post_signal_end_rows_used_for_features_or_scoring": False,
        "outcome_tail_used_only_for_existing_signal_windows": True,
        "model_retrained": False,
        "hyperparameters_changed": False,
        "selected_feature_set_changed": False,
        "candidate_threshold_changed": False,
        "calibration_refit": False,
        "signal_fraction_changed": False,
        "hard_started_filter_applied": False,
        "hard_breadth_filter_applied": False,
        "portfolio_backtest_performed": False,
        "transaction_costs_applied": False,
        "executable_exit_rule_claim_allowed": False,
    },
    "configuration_hash": stable_object_hash({
        "stage09_config": stage09_config,
        "stage04_policy": stage04_policy,
        "stage07_model_sha256": actual_model_hash,
        "stage08_policy": stage08_policy,
        "candidate_population_sha256": candidate_fingerprint,
        "inference_lock_file_sha256": lock_file_hash,
        "selected_signal_corrected_outcomes": selected_outcome_row.to_dict(),
    }),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "09_unseen_test_signal_evaluation_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Stage 09 v4 manifest:", manifest_path)
print("Manifest code SHA:", manifest["git_commit_sha"])


## 11. Final integrity checks and truthful interpretation


In [ ]:
assert candidate_errors_df.empty
assert outcome_errors_df.empty
assert corrected_outcome_errors_df.empty
assert market_index_errors_df.empty

assert total_test_rows == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_rows"]
)
assert total_test_eligible == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_eligible_events"]
)
assert len(evaluation_df) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert evaluation_df["event_id"].is_unique
assert evaluation_df["dEven"].between(
    TEST_START,
    SIGNAL_GENERATION_END,
).all()
assert evaluation_df["event_end_date"].le(
    SIGNAL_GENERATION_END
).all()
assert evaluation_df["meta_label"].isin([0, 1]).all()
assert np.isfinite(
    evaluation_df["xgboost_ranking_score"].to_numpy(dtype=float)
).all()

assert len(selected_evaluation_df) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)
assert selected_evaluation_df["event_id"].is_unique
assert selected_evaluation_df[
    "outcome_window_end_date"
].le(OUTCOME_OBSERVATION_TAIL_END).all()
assert np.isfinite(
    selected_evaluation_df[
        "corrected_event_return"
    ].to_numpy(dtype=float)
).all()
assert (
    selected_evaluation_df[
        "corrected_event_return"
    ].gt(0.0).astype(int)
    == selected_evaluation_df["meta_label"].astype(int)
).all()

assert len(BASE_MODEL_FEATURES) == 35
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert len(transformed_feature_names) == 47
assert actual_model_hash == stage07_manifest["model"]["model_sha256"]
assert actual_model_hash == stage09_config[
    "frozen_inputs"
]["expected_model_sha256"]

assert np.isclose(selected_fraction, 0.05)
assert minimum_per_date == 1
assert int(evaluation_df["selected_signal"].sum()) == int(
    stage09_config["frozen_inputs"]["expected_selected_signals"]
)

assert inference_lock_manifest[
    "outcome_columns_in_lock_file"
] is False
assert inference_lock_manifest[
    "test_labels_loaded_before_lock"
] is False
assert inference_lock_manifest[
    "test_outcomes_loaded_before_lock"
] is False

assert manifest["scientific_status"][
    "confirmatory_claim_allowed"
] is False
assert manifest["scientific_status"][
    "future_confirmation_required"
] is True
assert manifest["safeguards"][
    "scores_and_signals_locked_before_outcome_join"
] is True
assert manifest["safeguards"][
    "expected_performance_metrics_supplied_before_lock"
] is False
assert manifest["safeguards"][
    "expected_inference_lock_hash_supplied_before_run"
] is False
assert manifest["safeguards"]["model_retrained"] is False
assert manifest["safeguards"]["hyperparameters_changed"] is False
assert manifest["safeguards"][
    "selected_feature_set_changed"
] is False
assert manifest["safeguards"][
    "candidate_threshold_changed"
] is False
assert manifest["safeguards"]["calibration_refit"] is False
assert manifest["safeguards"]["signal_fraction_changed"] is False
assert manifest["safeguards"][
    "hard_started_filter_applied"
] is False
assert manifest["safeguards"][
    "hard_breadth_filter_applied"
] is False
assert manifest["safeguards"][
    "portfolio_backtest_performed"
] is False
assert manifest["safeguards"][
    "transaction_costs_applied"
] is False
assert manifest["safeguards"][
    "post_signal_end_rows_used_for_features_or_scoring"
] is False

assert signal_date_audit_df["selected_signals"].eq(
    signal_date_audit_df["required_signal_quota"]
).all()

print("Notebook 09 v4 integrity checks: PASSED")
print(
    "Scientific status:",
    "post-hoc retest on previously inspected period",
)
print("Confirmatory claim allowed: False")
print("Frozen model SHA256:", actual_model_hash)
print("Inference lock SHA256:", lock_file_hash)
print("Selected model:", SELECTED_MODEL)
print("Selected feature set:", SELECTED_FEATURE_SET)
print("Selected trial:", SELECTED_TRIAL)
print("Raw model features:", len(SELECTED_FEATURES))
print("Transformed model features:", len(transformed_feature_names))
print("Evaluation-period raw rows:", total_test_rows)
print("Evaluation-period eligible events:", total_test_eligible)
print("Long candidate events:", len(evaluation_df))
print("Signal dates:", evaluation_df["dEven"].nunique())
print("Frozen selected signals:", int(evaluation_df["selected_signal"].sum()))
print("Signal precision:", float(signal_classification_metrics_df.iloc[0]["precision"]))
print("Signal specificity:", float(signal_classification_metrics_df.iloc[0]["specificity"]))
print("Corrected selected-signal win rate:", float(selected_outcome_row["win_rate"]))
print("Corrected selected-signal payoff ratio:", float(selected_outcome_row["payoff_ratio"]))
print("Corrected selected-signal profit factor:", float(selected_outcome_row["profit_factor"]))
print("Corrected selected-signal mean event return:", float(selected_outcome_row["mean_corrected_event_return"]))
print("Scores locked before outcomes: True")
print("Expected performance values supplied before lock: False")
print("Portfolio backtest performed: False")
print("Transaction costs applied: False")
print(
    "Corrected return interpretation: gross selected-signal event outcome; "
    "upper outcomes use an ex-post 30-observation maximum and are not "
    "executable portfolio returns."
)
print(
    "Next action: independently audit this post-hoc retest, then use later "
    "unseen data or prospective paper trading for genuine confirmation."
)


## Review before freezing Stage 09 v4

Inspect all `09_` audit, manifest, and prediction files.

This run must be described as a **post-hoc retest**, not confirmatory unseen-test
validation. Do not promote the corrected event-return metrics to executable
portfolio performance. Genuine confirmation requires later untouched data or
prospective paper trading.
